# 05 — Training: Putting It All Together

We now combine everything from notebooks 01-04 into a full GPT-style language model, train it on a small text corpus, and generate text.

## Table of Contents
1. [Model Configuration](#1-model-configuration)
2. [Loading the Model](#2-loading-the-model)
3. [Training Data Preparation](#3-training-data-preparation)
4. [Training Loop](#4-training-loop)
5. [Text Generation](#5-text-generation)
6. [Key Takeaways](#6-key-takeaways)

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
import tiktoken
import time
import sys, os
from tqdm.auto import tqdm

sys.path.insert(0, os.path.abspath('.'))
from utils.model import GPTModel, DEFAULT_CONFIG
from utils.visualisation import plot_loss_curves

torch.manual_seed(42)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

# Use GPU if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 
                      'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Model Configuration

We keep the model tiny for CPU training. All hyperparameters are in a single config dict — change this to scale up later.

In [ ]:
# ============================================================
# Hyperparameter Configuration
# ============================================================
# Change these values to scale the model up or down.
# The tiny defaults below train in ~2-5 minutes on CPU.

config = {
    # Model architecture
    "vocab_size":   50257,   # GPT-2 tokenizer vocabulary size (tiktoken)
    "d_model":      64,      # Embedding / hidden dimension
    "n_heads":      4,       # Number of attention heads (d_model must be divisible)
    "n_layers":     2,       # Number of transformer blocks
    "d_ff":         256,     # Feed-forward inner dimension (typically 4 × d_model)
    "max_seq_len":  128,     # Maximum context window length
    "dropout":      0.1,     # Dropout probability
    
    # Training
    "batch_size":   16,      # Sequences per training step
    "learning_rate": 3e-4,   # Peak learning rate
    "num_epochs":   20,      # Number of passes over the training data
    "warmup_steps": 100,     # Linear warmup steps before cosine decay
    "val_split":    0.1,     # Fraction of data held out for validation
    "eval_interval": 50,     # Evaluate every N steps
}

print("="*60)
print("MODEL CONFIGURATION")
print("="*60)
for k, v in config.items():
    print(f"  {k:>20}: {v}")

## 2. Loading the Model

We use the `GPTModel` class from `utils/model.py` (built from the components in notebooks 02-04).

In [ ]:
# ============================================================
# Instantiate the model
# ============================================================

model = GPTModel(config).to(device)

n_params = model.count_parameters()
print(f"Model created on {device}")
print(f"Total trainable parameters: {n_params:,}")
print(f"")

# Breakdown by component
print("Parameter breakdown:")
print(f"  Token embedding:   {model.token_embedding.weight.numel():>10,}")
for i, block in enumerate(model.blocks):
    block_params = sum(p.numel() for p in block.parameters())
    print(f"  Block {i}:            {block_params:>10,}")
ln_params = sum(p.numel() for p in model.ln_final.parameters())
print(f"  Final LayerNorm:   {ln_params:>10,}")
print(f"  LM head (tied):    (shared with token embedding)")
print(f"  {'Total':>18}: {n_params:>10,}")

## 3. Training Data Preparation

We'll use a small corpus of text about engineering and science. For a toy model, we don't need much — just enough to demonstrate learning.

In [ ]:
# ============================================================
# Training corpus (inline text)
# ============================================================
# A small but coherent body of text for our toy model to learn.
# The model will learn patterns in this text and try to generate
# similar text. With such a small model, don't expect Shakespeare —
# but it WILL learn the basic patterns.

corpus = """
Engineering is the application of scientific principles to design and build machines, structures, and systems. Engineers use mathematics, physics, and computational tools to solve complex problems. The field of engineering encompasses many disciplines including mechanical, electrical, civil, and software engineering.

A control system is a system that manages, commands, directs, or regulates the behaviour of other systems. Control systems are found in many applications from simple household thermostats to complex industrial automation systems. The fundamental concept is feedback, where the output of a system is measured and compared to the desired setpoint.

Embedded systems are computer systems designed to perform specific functions within larger mechanical or electrical systems. They are found in automobiles, medical devices, industrial equipment, and consumer electronics. An embedded system typically consists of a microcontroller, memory, input and output interfaces, and specialised software.

Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing algorithms that can access data, learn from it, and make predictions or decisions. Neural networks, inspired by the human brain, are a fundamental architecture in modern machine learning.

A transformer is a neural network architecture that uses self-attention mechanisms to process sequential data. Unlike recurrent neural networks, transformers can process all positions in a sequence simultaneously, making them highly parallelisable. The key innovation is the attention mechanism, which allows each element in a sequence to attend to all other elements, capturing long-range dependencies effectively.

Functional safety is the part of the overall safety of a system that depends on the system or equipment operating correctly in response to its inputs. In automotive systems, functional safety standards like ISO 26262 provide guidelines for ensuring that electronic systems in vehicles operate safely. Safety integrity levels define the required level of risk reduction.

Robotics combines engineering and computer science to design, build, and program robots. A robot typically includes sensors for perceiving the environment, actuators for interacting with it, and a control system that coordinates its actions. Modern robots use artificial intelligence to adapt to changing environments and perform complex tasks autonomously.

Software engineering applies systematic approaches to the development, operation, and maintenance of software systems. Version control systems like Git track changes to source code over time. Testing frameworks verify that software behaves correctly. Continuous integration and deployment pipelines automate the build, test, and release process.

Signal processing is the analysis, modification, and synthesis of signals such as sound, images, and sensor data. Digital signal processing uses mathematical algorithms to manipulate discrete-time signals. Common operations include filtering, spectral analysis using the Fourier transform, and compression. Signal processing is essential in communications, audio engineering, and control systems.

Systems engineering is an interdisciplinary approach to designing, integrating, and managing complex systems over their life cycles. It focuses on defining stakeholder needs, establishing requirements, and synthesising system-level solutions. Model-based systems engineering uses models rather than documents as the primary means of information exchange.
"""

print(f"Corpus length: {len(corpus)} characters")
print(f"Corpus preview: {corpus[:200]}...")

In [ ]:
# ============================================================
# Tokenise the corpus using tiktoken (GPT-2 tokenizer)
# ============================================================

enc = tiktoken.get_encoding("gpt2")

# Encode the entire corpus to token IDs
token_ids = enc.encode(corpus)
data = torch.tensor(token_ids, dtype=torch.long)

print(f"Total tokens: {len(data):,}")
print(f"Unique tokens used: {len(set(token_ids))}")
print(f"Vocabulary size: {config['vocab_size']:,}")
print(f"")
print(f"First 20 tokens (IDs):  {data[:20].tolist()}")
print(f"First 20 tokens (text): {[enc.decode([t]) for t in data[:20].tolist()]}")

In [ ]:
# ============================================================
# Train/validation split
# ============================================================

val_size = int(len(data) * config["val_split"])
train_data = data[:-val_size]
val_data = data[-val_size:]

print(f"Training tokens:   {len(train_data):,}")
print(f"Validation tokens: {len(val_data):,}")

In [ ]:
# ============================================================
# Batch generation function
# ============================================================
# Creates random (input, target) pairs from the token sequence.
# For next-token prediction:
#   input  = tokens[i : i + seq_len]
#   target = tokens[i+1 : i + seq_len + 1]  (shifted by 1)

def get_batch(data, batch_size, seq_len, device):
    """
    Generate a random batch of (input, target) pairs.
    
    Args:
        data:       1-D tensor of token IDs
        batch_size: number of sequences in the batch
        seq_len:    context window length
        device:     torch device
    
    Returns:
        x: (batch_size, seq_len)     — input token IDs
        y: (batch_size, seq_len)     — target token IDs (shifted right by 1)
    """
    # Random starting indices
    max_start = len(data) - seq_len - 1
    starts = torch.randint(0, max_start, (batch_size,))
    
    x = torch.stack([data[s : s + seq_len] for s in starts])        # (batch, seq)
    y = torch.stack([data[s + 1 : s + seq_len + 1] for s in starts])  # (batch, seq)
    
    return x.to(device), y.to(device)

# Test it
x_batch, y_batch = get_batch(train_data, batch_size=2, seq_len=8, device=device)
print(f"Input batch:  {x_batch.shape}")
print(f"Target batch: {y_batch.shape}")
print(f"\nExample (first sequence):")
print(f"  Input:  {x_batch[0].tolist()}")
print(f"  Target: {y_batch[0].tolist()}")
print(f"  Notice: target is input shifted right by 1 (next token prediction)")

## 4. Training Loop

In [ ]:
# ============================================================
# Learning rate schedule: linear warmup + cosine decay
# ============================================================

def get_lr(step, warmup_steps, max_steps, max_lr, min_lr=1e-5):
    """
    Compute learning rate at a given step.
    
    Phase 1 (warmup): Linear increase from 0 to max_lr
    Phase 2 (decay):  Cosine decrease from max_lr to min_lr
    """
    if step < warmup_steps:
        # Linear warmup
        return max_lr * (step / warmup_steps)
    else:
        # Cosine decay
        progress = (step - warmup_steps) / max(1, max_steps - warmup_steps)
        return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

# Visualise the LR schedule
max_steps = 1000
lrs = [get_lr(s, config["warmup_steps"], max_steps, config["learning_rate"]) for s in range(max_steps)]

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(lrs, linewidth=2)
ax.set_xlabel('Step')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedule: Warmup + Cosine Decay', fontweight='bold')
ax.axvline(x=config["warmup_steps"], color='red', linestyle='--', alpha=0.5, label=f'Warmup ends ({config["warmup_steps"]} steps)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Evaluation function
# ============================================================

@torch.no_grad()
def evaluate(model, data, batch_size, seq_len, device, n_batches=10):
    """Compute average loss over n_batches of random samples."""
    model.eval()
    total_loss = 0.0
    for _ in range(n_batches):
        x, y = get_batch(data, batch_size, seq_len, device)
        logits = model(x)  # (batch, seq, vocab_size)
        # Flatten for cross-entropy: (batch*seq, vocab_size) vs (batch*seq,)
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            y.view(-1)
        )
        total_loss += loss.item()
    model.train()
    return total_loss / n_batches

In [ ]:
# ============================================================
# TRAINING LOOP
# ============================================================

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"], weight_decay=0.01)

# Calculate total steps
tokens_per_epoch = len(train_data) - config["max_seq_len"] - 1
batches_per_epoch = max(1, tokens_per_epoch // (config["batch_size"] * config["max_seq_len"]))
total_steps = batches_per_epoch * config["num_epochs"]

print("="*60)
print("TRAINING")
print("="*60)
print(f"  Training tokens:  {len(train_data):,}")
print(f"  Batches/epoch:    {batches_per_epoch}")
print(f"  Total steps:      {total_steps}")
print(f"  Device:           {device}")
print("="*60)

# Training history
train_losses = []
val_losses = []
val_steps = []

model.train()
start_time = time.time()
step = 0

for epoch in range(config["num_epochs"]):
    epoch_loss = 0.0
    
    for batch_idx in range(batches_per_epoch):
        # Get batch
        x, y = get_batch(train_data, config["batch_size"], config["max_seq_len"], device)
        
        # Update learning rate
        lr = get_lr(step, config["warmup_steps"], total_steps, config["learning_rate"])
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr
        
        # Forward pass
        logits = model(x)  # (batch, seq, vocab_size)
        
        # Cross-entropy loss
        # Reshape: (batch*seq, vocab_size) vs (batch*seq,)
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            y.view(-1)
        )
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping (prevents exploding gradients)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # Update weights
        optimizer.step()
        
        # Record
        train_losses.append(loss.item())
        epoch_loss += loss.item()
        
        # Periodic evaluation
        if step % config["eval_interval"] == 0:
            val_loss = evaluate(model, val_data, config["batch_size"],
                              config["max_seq_len"], device)
            val_losses.append(val_loss)
            val_steps.append(step)
        
        step += 1
    
    # Epoch summary
    avg_epoch_loss = epoch_loss / batches_per_epoch
    elapsed = time.time() - start_time
    print(f"  Epoch {epoch+1:>2}/{config['num_epochs']}  |  "
          f"Loss: {avg_epoch_loss:.4f}  |  "
          f"LR: {lr:.6f}  |  "
          f"Time: {elapsed:.1f}s")

total_time = time.time() - start_time
print(f"\n{'='*60}")
print(f"Training complete in {total_time:.1f}s")
print(f"Final training loss: {train_losses[-1]:.4f}")
print(f"Final validation loss: {val_losses[-1]:.4f}")
print(f"{'='*60}")

In [ ]:
# ============================================================
# Plot training curves
# ============================================================

# For val losses, we need to expand to match the train loss x-axis
# Create a sparse val_loss array aligned with training steps
plot_loss_curves(
    train_losses,
    val_losses=None,  # Plot separately below for clarity
    title="Training Loss",
    smoothing=0.9
)

# Separate validation plot
if val_losses:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(val_steps, val_losses, 'o-', color='tomato', linewidth=2, label='Validation')
    ax.set_title('Validation Loss', fontweight='bold', fontsize=13)
    ax.set_xlabel('Step')
    ax.set_ylabel('Cross-Entropy Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# Save the trained model checkpoint
# ============================================================
# This will be loaded by notebook 06 for attention visualisation.

checkpoint = {
    "model_state_dict": model.state_dict(),
    "config": config,
    "train_losses": train_losses,
    "val_losses": val_losses,
    "final_train_loss": train_losses[-1],
    "final_val_loss": val_losses[-1] if val_losses else None,
}

os.makedirs("checkpoints", exist_ok=True)
save_path = "checkpoints/model.pt"
torch.save(checkpoint, save_path)

file_size = os.path.getsize(save_path) / (1024 * 1024)
print(f"Model saved to: {save_path}")
print(f"Checkpoint size: {file_size:.2f} MB")

## 5. Text Generation

Now let's generate text from our trained model! We'll try different generation strategies:
- **Greedy**: Always pick the most probable next token
- **Temperature sampling**: Control randomness (higher = more creative, lower = more deterministic)
- **Top-k sampling**: Only sample from the top-k most probable tokens

In [ ]:
# ============================================================
# Text generation helper
# ============================================================

def generate_text(model, prompt, enc, max_new_tokens=100, temperature=1.0,
                  top_k=0, greedy=False, device='cpu'):
    """
    Generate text from a prompt using the trained model.
    
    Args:
        model:          trained GPTModel
        prompt:         string to start generation from
        enc:            tiktoken encoding
        max_new_tokens: how many tokens to generate
        temperature:    sampling temperature (1.0 = normal, <1 = focused, >1 = diverse)
        top_k:          if >0, only sample from top-k tokens
        greedy:         if True, always pick argmax
        device:         torch device
    
    Returns:
        generated text string
    """
    # Encode prompt
    prompt_ids = enc.encode(prompt)
    ids = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    
    # Generate
    output_ids = model.generate(
        ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        greedy=greedy
    )
    
    # Decode back to text
    return enc.decode(output_ids[0].tolist())

In [ ]:
# ============================================================
# Generate with different strategies
# ============================================================

prompt = "Engineering is"

print("="*70)
print(f"PROMPT: \"{prompt}\"")
print("="*70)

# Greedy (deterministic)
print("\n--- Greedy (always pick most probable) ---")
text = generate_text(model, prompt, enc, max_new_tokens=80, greedy=True, device=device)
print(text)

# Temperature = 0.7 (slightly creative)
print("\n--- Temperature = 0.7 (balanced) ---")
text = generate_text(model, prompt, enc, max_new_tokens=80, temperature=0.7, device=device)
print(text)

# Temperature = 1.2 (more creative)
print("\n--- Temperature = 1.2 (more diverse) ---")
text = generate_text(model, prompt, enc, max_new_tokens=80, temperature=1.2, device=device)
print(text)

# Top-k = 10
print("\n--- Top-k = 10 (sample from top 10 tokens) ---")
text = generate_text(model, prompt, enc, max_new_tokens=80, top_k=10, temperature=0.8, device=device)
print(text)

In [ ]:
# ============================================================
# Try different prompts
# ============================================================

prompts = [
    "A control system",
    "Machine learning is",
    "The transformer architecture",
    "Functional safety",
]

for prompt in prompts:
    print(f"\n{'='*60}")
    print(f"PROMPT: \"{prompt}\"")
    print(f"{'='*60}")
    text = generate_text(model, prompt, enc, max_new_tokens=60, 
                        temperature=0.7, top_k=20, device=device)
    print(text)

print("\n" + "="*60)
print("NOTE: This is a tiny model trained on very little data.")
print("The generated text may be repetitive or nonsensical.")
print("With more data and a larger model, quality improves dramatically.")
print("="*60)

## 6. Key Takeaways

---

### What We Built

A complete GPT-style language model from scratch:

```
Token IDs → Embedding → Positional Encoding
    → N × TransformerBlock (Attention + FFN + Residual + LayerNorm)
    → LayerNorm → Linear Head → Logits → Softmax → Next Token
```

### Training Details

| Aspect | Choice | Why |
|--------|--------|-----|
| **Loss** | Cross-entropy | Standard for classification (next token from vocabulary) |
| **Optimizer** | AdamW | Adam with decoupled weight decay — standard for transformers |
| **LR schedule** | Warmup + cosine decay | Warmup prevents early instability; decay prevents overfitting |
| **Gradient clipping** | max_norm=1.0 | Prevents exploding gradients in attention layers |
| **Weight tying** | Embedding = LM head | Fewer parameters, better generalisation |

### Scaling Up

To make this model better, change the config dict:
- **More data**: Use a larger corpus (e.g., Wikipedia, books)
- **Larger model**: Increase d_model (128→768), n_layers (2→12), n_heads (4→12)
- **Longer context**: Increase max_seq_len (128→1024)
- **GPU training**: Move to CUDA for 10-100× speedup

### What's Next

In the next notebook (**06 — Visualising Attention**), we'll load this trained model and explore what the attention heads actually learned.

---